In [78]:
import requests 
from bs4 import BeautifulSoup
import pandas as pd



Task 1: Capture title,price,star_rating,availability and category of atleast 60 books. 


In [79]:
from urllib.parse import urljoin

url = "http://books.toscrape.com/"

books = []

for page_number in range(1, 6):
    page_url = url if page_number == 1 else f"{url}catalogue/page-{page_number}.html"
    response = requests.get(page_url, timeout=30)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, "html.parser")
    all_books = soup.select("article.product_pod")
    for book in all_books:
        title = book.h3.a["title"]
        price = book.find("p", class_="price_color").text.strip()
        rating = book.find("p", class_="star-rating")["class"][1]
        availability = book.find("p", class_="instock availability").text.strip()

        book_link = book.h3.a["href"]
        complete_book_url = urljoin(page_url, book_link)
        book_url_response = requests.get(complete_book_url, timeout=30)
        book_url_response.raise_for_status()
        book_url_response_soup = BeautifulSoup(book_url_response.content, "html.parser")
        breadcrumb = book_url_response_soup.find("ul", class_="breadcrumb")
        category = breadcrumb.find_all("a")[2].text.strip()
        books.append({
            "title": title,
            "price": price,
            "rating": rating,
            "availability": availability,
            "category": category,
        })

print(f"Scraped {len(books)} books across {len({book['category'] for book in books})} categories.")
assert len(books) >= 60, "The scrape must produce at least 60 books."

Scraped 100 books across 29 categories.


Task 2: Clean the scraped fields into proper types

In [81]:
rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
df = pd.DataFrame(books)
df["price_gbp"] = pd.to_numeric(
    df["price"].str.replace("£", "", regex=False).str.strip(),
    errors="coerce"
)
df["rating"] = df["rating"].map(rating_map)

# The source uses "In stock" or "Out of stock". Unexpected availability text
# is treated as an unparseable row and dropped rather than silently misclassified.
valid_availability = df["availability"].str.contains(
    r"^(?:In stock|Out of stock)", case=False, na=False
)
invalid_availability_count = int((~valid_availability).sum())
df = df.loc[valid_availability].copy()
df["in_stock"] = df["availability"].str.startswith("In stock", na=False)

# Numeric parse failures use median imputation so valid book records are retained.
if df["price_gbp"].isna().any():
    df["price_gbp"] = df["price_gbp"].fillna(df["price_gbp"].median())

if df["rating"].isna().any():
    df["rating"] = df["rating"].fillna(round(df["rating"].median()))
df["rating"] = df["rating"].astype(int)

print(f"Dropped {invalid_availability_count} rows with unrecognized availability text.")
print(df[["price_gbp", "rating", "in_stock"]].dtypes)

Dropped 0 rows with unrecognized availability text.
price_gbp    float64
rating         int64
in_stock        bool
dtype: object


Task 3: Convert GBP to INR using the project's fixed baseline rate.

In [82]:
# Required project-defined conversion: 1 GBP = 105.50 INR.
GBP_TO_INR_RATE = 105.50
df["price_inr"] = df["price_gbp"] * GBP_TO_INR_RATE
assert (df["price_inr"] == df["price_gbp"] * 105.50).all()

Task 4: Design a normalized SQLite schema with at least two tables sharing a primary/foreign key relationship

In [83]:
import sqlite3

conn = sqlite3.connect("books.db")
conn.execute("PRAGMA foreign_keys = ON")
cursor = conn.cursor()

# Drop tables if they already exist (useful for re-running the cell).
cursor.execute("DROP TABLE IF EXISTS books")
cursor.execute("DROP TABLE IF EXISTS categories")

# Create categories table.
cursor.execute("""
CREATE TABLE categories (
    category_id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_name TEXT UNIQUE NOT NULL
)
""")

# Create books table with a foreign key reference to categories.
cursor.execute("""
CREATE TABLE books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT NOT NULL,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    in_stock INTEGER,
    category_id INTEGER,
    FOREIGN KEY (category_id) REFERENCES categories(category_id)
)
""")

conn.commit()

Task 5: Insert the cleaned, converted book data into SQLite and execute five SQL queries demonstrating filtering, sorting, limiting, distinct values, range/membership conditions, and a category JOIN.


In [84]:
# Insert unique categories
categories = df['category'].unique().tolist()
cursor.executemany(
    "INSERT INTO categories (category_name) VALUES (?)",
    [(c,) for c in categories]
)
conn.commit()

# Build a mapping from category_name to category_id
cursor.execute("SELECT category_id, category_name FROM categories")
category_map = {name: cid for cid, name in cursor.fetchall()}

In [85]:

# Insert books
book_rows = [
    (
        row['title'],
        row['price_gbp'],
        row['price_inr'],
        int(row['rating']),
        int(row['in_stock']),
        category_map[row['category']]
    )
    for _, row in df.iterrows()
]

cursor.executemany(
    """
    INSERT INTO books (title, price_gbp, price_inr, rating, in_stock, category_id)
    VALUES (?, ?, ?, ?, ?, ?)
    """,
    book_rows
)
conn.commit()


In [ ]:
cursor.execute("SELECT * FROM books")
tables = cursor.fetchall()
for table in tables:
    print(table)

In [ ]:
cursor.execute("SELECT * FROM categories")
tabless = cursor.fetchall()
for table in tabless:
    print(table)

In [86]:
# --- Query 1: SELECT / WHERE / ORDER BY / LIMIT ---
query1 = """
SELECT title, price_gbp, rating
FROM books
WHERE rating >= 4
ORDER BY price_gbp DESC
LIMIT 10
"""
cursor.execute(query1)
result1 = cursor.fetchall()
print("Query 1: Top 10 highest priced books with rating >= 4")
for row in result1:
    print(row)


Query 1: Top 10 highest priced books with rating >= 4
('The Death of Humanity: and the Case for Life', 58.11, 4)
('The Past Never Ends', 56.5, 4)
('Sapiens: A Brief History of Humankind', 54.23, 5)
("Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)", 52.29, 5)
('Behind Closed Doors', 52.22, 4)
('We Love You, Charlie Freeman', 50.27, 5)
('Sharp Objects', 47.82, 4)
('Private Paris (Private #10)', 47.61, 5)
('Wall and Piece', 44.18, 4)
('Unseen City: The Majesty of Pigeons, the Discreet Charm of Snails & Other Wonders of the Urban Wilderness', 44.18, 4)


In [87]:
# --- Query 2: DISTINCT ---
query2 = """
SELECT DISTINCT category_id
FROM books
WHERE in_stock = 1
"""
cursor.execute(query2)
result2 = cursor.fetchall()
print("\nQuery 2: Distinct category_ids with in-stock books")
category_ids = [row[0] for row in result2]
for row in category_ids:
    print(row)


Query 2: Distinct category_ids with in-stock books
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29


In [88]:
# --- Query 3: IN ---
query3 = """
SELECT title, rating, price_gbp
FROM books
WHERE rating IN (1, 5)
ORDER BY rating, title
LIMIT 15
"""
cursor.execute(query3)
result3 = cursor.fetchall()
print("\nQuery 3: Books with rating 1 or 5")
for row in result3:
    print(row)


Query 3: Books with rating 1 or 5
('In Her Wake', 1, 12.84)
('In a Dark, Dark Wood', 1, 19.63)
('Layered: Baking, Building, and Styling Spectacular Cakes', 1, 40.11)
('Mesaerion: The Best Science Fiction Stories 1800-1849', 1, 37.59)
('Olio', 1, 23.88)
('Online Marketing for Busy Authors: A Step-By-Step Guide', 1, 46.35)
('Pop Gun War, Volume 1: Gift', 1, 18.97)
('Soumission', 1, 50.1)
('The Age of Genius: The Seventeenth Century and the Birth of the Modern Mind', 1, 19.73)
('The Bear and the Piano', 1, 36.89)
('The Black Maria', 1, 52.15)
('The Electric Pencil: Drawings from Inside State Hospital No. 3', 1, 56.06)
('The Gutsy Girl: Escapades for Your Life of Epic Adventure', 1, 37.13)
('The Pioneer Woman Cooks: Dinnertime: Comfort Classics, Freezer Food, 16-Minute Meals, and Other Delicious Ways to Solve Supper!', 1, 56.41)
('The Requiem Red', 1, 22.65)


In [89]:
# --- Query 4: BETWEEN ---
query4 = """
SELECT title, price_gbp
FROM books
WHERE price_gbp BETWEEN 20 AND 30
ORDER BY price_gbp
"""
cursor.execute(query4)
result4 = cursor.fetchall()
print("\nQuery 4: Books priced between £20 and £30")
for row in result4:
    print(row)


Query 4: Books priced between £20 and £30
('The Inefficiency Assassin: Time Management Tactics for Working Smarter, Not Longer', 20.59)
("Shakespeare's Sonnets", 20.66)
('In the Country We Love: My Family Divided', 22.0)
("America's Cradle of Quarterbacks: Western Pennsylvania's Football Factory from Johnny Unitas to Joe Montana", 22.5)
('The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics', 22.6)
('The Requiem Red', 22.65)
('#HigherSelfie: Wake Up Your Life. Free Your Soul. Find Your Tribe.', 23.11)
('The Elephant Tree', 23.82)
('Olio', 23.88)
('The Mindfulness and Acceptance Workbook for Anxiety: A Guide to Breaking Free from Anxiety, Phobias, and Worry Using Acceptance and Commitment Therapy', 23.89)
('Saga, Volume 6 (Saga (Collected Editions) #6)', 25.02)
('Chase Me (Paris Nights #2)', 25.27)
('Unbound: How Eight Technologies Made Us Human, Transformed Society, and Brought Our World to the Brink', 25.52)
('Reasons to Stay Alive', 26.41)


In [90]:
# --- Query 5: JOIN (10 highest-rated books per category example: top 10 overall with category name) ---
query5 = """
SELECT b.title, b.rating, b.price_gbp, c.category_name
FROM books b
JOIN categories c ON b.category_id = c.category_id
ORDER BY b.rating DESC, b.price_gbp DESC
LIMIT 10
"""
cursor.execute(query5)
result5 = cursor.fetchall()
print("\nQuery 5: Top 10 highest-rated books with category name (JOIN)")
for row in result5:
    print(row)


Query 5: Top 10 highest-rated books with category name (JOIN)
('Sapiens: A Brief History of Humankind', 5, 54.23, 'History')
("Scott Pilgrim's Precious Little Life (Scott Pilgrim #1)", 5, 52.29, 'Sequential Art')
('We Love You, Charlie Freeman', 5, 50.27, 'Fiction')
('Private Paris (Private #10)', 5, 47.61, 'Fiction')
('Worlds Elsewhere: Journeys Around Shakespeare’s Globe', 5, 40.3, 'Nonfiction')
('Join', 5, 35.67, 'Science Fiction')
('Rip it Up and Start Again', 5, 35.02, 'Music')
('Black Dust', 5, 34.53, 'Romance')
("The Activist's Tao Te Ching: Ancient Advice for a Modern Revolution", 5, 32.24, 'Spirituality')
('Chase Me (Paris Nights #2)', 5, 25.27, 'Romance')


Task 6: Read query results into pandas with `pd.read_sql` and reproduce the JOIN result with `pd.merge` using the in-memory DataFrames.

In [91]:
import json

# Read two saved SQL query results into pandas DataFrames.
query1_df = pd.read_sql(query1, conn)
query4_df = pd.read_sql(query4, conn)

print("Query 1 result loaded with pd.read_sql:")
display(query1_df)
print("Query 4 result loaded with pd.read_sql:")
display(query4_df)

# Reproduce Query 5's JOIN using only the in-memory pandas DataFrames.
books_for_merge = df.copy()
books_for_merge["category_id"] = books_for_merge["category"].map(category_map)
categories_df = pd.DataFrame(
    list(category_map.items()),
    columns=["category_name", "category_id"]
)

merge_result = (
    books_for_merge.merge(categories_df, on="category_id", how="inner")
    [["title", "rating", "price_gbp", "category_name"]]
    .sort_values(["rating", "price_gbp"], ascending=[False, False])
    .head(10)
    .reset_index(drop=True)
)

# Read the SQL JOIN result and align column names/order for comparison.
sql_join_result = (
    pd.read_sql(query5, conn)
    [["title", "rating", "price_gbp", "category_name"]]
    .reset_index(drop=True)
)

print("SQL JOIN result:")
display(sql_join_result)
print("Equivalent pandas.merge result:")
display(merge_result)
print("JOIN outputs equivalent:", sql_join_result.equals(merge_result))

# Persist every query string together with its tabular output for review.
query_outputs = {}
for query_name, query_text in {
    "query1": query1,
    "query2": query2,
    "query3": query3,
    "query4": query4,
    "query5": query5,
}.items():
    output_df = pd.read_sql(query_text, conn)
    query_outputs[query_name] = {
        "query": query_text.strip(),
        "output": output_df.to_dict(orient="records"),
    }

with open("query_outputs.json", "w", encoding="utf-8") as output_file:
    json.dump(query_outputs, output_file, indent=2, default=str)

print("Saved query strings and outputs to query_outputs.json")

Query 1 result loaded with pd.read_sql:


,title,price_gbp,rating
0,The Death of Humanity: and the Case for Life,58.11,4
1,The Past Never Ends,56.50,4
2,Sapiens: A Brief History of Humankind,54.23,5
3,Scott Pilgrim's Precious Little Life (Scott Pi...,52.29,5
4,Behind Closed Doors,52.22,4
5,"We Love You, Charlie Freeman",50.27,5
6,Sharp Objects,47.82,4
7,Private Paris (Private #10),47.61,5
8,Wall and Piece,44.18,4
9,"Unseen City: The Majesty of Pigeons, the Discr...",44.18,4


Query 4 result loaded with pd.read_sql:


,title,price_gbp
0,The Inefficiency Assassin: Time Management Tac...,20.59
1,Shakespeare's Sonnets,20.66
2,In the Country We Love: My Family Divided,22.00
3,America's Cradle of Quarterbacks: Western Penn...,22.50
4,The Boys in the Boat: Nine Americans and Their...,22.60
5,The Requiem Red,22.65
6,#HigherSelfie: Wake Up Your Life. Free Your So...,23.11
7,The Elephant Tree,23.82
8,Olio,23.88
9,The Mindfulness and Acceptance Workbook for An...,23.89


SQL JOIN result:


,title,rating,price_gbp,category_name
0,Sapiens: A Brief History of Humankind,5,54.23,History
1,Scott Pilgrim's Precious Little Life (Scott Pi...,5,52.29,Sequential Art
2,"We Love You, Charlie Freeman",5,50.27,Fiction
3,Private Paris (Private #10),5,47.61,Fiction
4,Worlds Elsewhere: Journeys Around Shakespeare’...,5,40.30,Nonfiction
5,Join,5,35.67,Science Fiction
6,Rip it Up and Start Again,5,35.02,Music
7,Black Dust,5,34.53,Romance
8,The Activist's Tao Te Ching: Ancient Advice fo...,5,32.24,Spirituality
9,Chase Me (Paris Nights #2),5,25.27,Romance


Equivalent pandas.merge result:


,title,rating,price_gbp,category_name
0,Sapiens: A Brief History of Humankind,5,54.23,History
1,Scott Pilgrim's Precious Little Life (Scott Pi...,5,52.29,Sequential Art
2,"We Love You, Charlie Freeman",5,50.27,Fiction
3,Private Paris (Private #10),5,47.61,Fiction
4,Worlds Elsewhere: Journeys Around Shakespeare’...,5,40.30,Nonfiction
5,Join,5,35.67,Science Fiction
6,Rip it Up and Start Again,5,35.02,Music
7,Black Dust,5,34.53,Romance
8,The Activist's Tao Te Ching: Ancient Advice fo...,5,32.24,Spirituality
9,Chase Me (Paris Nights #2),5,25.27,Romance


JOIN outputs equivalent: True
Saved query strings and outputs to query_outputs.json
